In [10]:
from google.colab import drive
drive.mount('/content/drive')


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
pip install noisereduce

In [ ]:
pip install surfboard

## Imports & Functions:

In [ ]:
import librosa
import numpy as np
import librosa, librosa.display
import matplotlib.pyplot as plt
import math
import os
import noisereduce as nr
import pandas as pd
from tqdm import tqdm
import copy
from surfboard.sound import Waveform
from surfboard.feature_extraction import extract_features
from scipy.io.wavfile import write

from scipy.stats import kurtosis , skew
def features(MFCCs , filter_banks , silence_percentage , sil_start , sil_stop , my_jitters , my_shimmers , my_HNR , my_cor) :
  MFCC2 = max(MFCCs[2-1,:])

  kurt_MFCC30 = kurtosis(MFCCs[30-1,:])

  mean_MFCC30 = np.mean(MFCCs[30-1,:])

  skew_MFCC2 = skew(MFCCs[2-1,:])

  mean_MFCC16 = np.mean(MFCCs[16-1,:])

  flt_bnk_eng22 = sum(filter_banks[22-1,:])

  # Spectral centroid -C14

  MFCC30 = max(MFCCs[30-1,:])

  kurt_MFCC16 = kurtosis(MFCCs[16-1,:])

  flt_bnk_eng2 = sum(filter_banks[2-1,:])

  flt_bnk_eng24 = sum(filter_banks[24-1,:])

  MFCC1 = max(MFCCs[1-1,:])

  flt_bnk_eng15 = sum(filter_banks[15-1,:])

  kurt_MFCC2 = kurtosis(MFCCs[2-1,:])

  flt_bnk_eng20 = sum(filter_banks[20-1,:])

  flt_bnk_eng13 = sum(filter_banks[13-1,:])

  n_sil_segments = len(sil_start)

  frac_silence = silence_percentage
  try:
    min_sil_len = min((sil_stop - sil_start))/sr
  except:
    min_sil_len = (sil_stop[0] - sil_start[0])/sr

  jit = my_jitters
  shim = my_shimmers
  hnr = my_HNR 
  cor = my_cor
  return [MFCC2 , kurt_MFCC30 , mean_MFCC30 , skew_MFCC2 , mean_MFCC16 , flt_bnk_eng22 , MFCC30 , kurt_MFCC16 , flt_bnk_eng2 , flt_bnk_eng24 , MFCC1 , flt_bnk_eng15 , kurt_MFCC2 , flt_bnk_eng20 , flt_bnk_eng13 , n_sil_segments , frac_silence , min_sil_len , jit , shim , hnr , cor]

def getEnvelope(inputSignal):  
    # Taking the absolute value 
    absoluteSignal = []
    for sample in inputSignal:
        absoluteSignal.append (abs (sample))
    
    # Peak detection
    intervalLength = 50
    outputSignal = []
    
    for baseIndex in range (intervalLength, len (absoluteSignal)):
        maximum = 0
        for lookbackIndex in range (intervalLength):
            maximum = max (absoluteSignal [baseIndex - lookbackIndex], maximum)
        outputSignal.append (maximum)
    return outputSignal

def silence_detection(signal_env,mod):
    n1 = 0.95 # percent of min amp values to calculate th
    k1 = 1.1 # constant coefficient of th
    signal_pos_sort = np.array(sorted(signal_env))
    I0 = np.argsort(signal_env)
    n_percent = math.floor(len(signal_pos_sort)*n1)
    th = k1*np.mean(signal_pos_sort[0:n_percent])


    ind_silence = list(np.argwhere(abs(signal_env)>th).T)
    ind_silence = list(ind_silence[0])
    ind_silence2 = ind_silence[1:]
    ind_silence2.append(ind_silence[-1])
    ind_silence3 = ind_silence[:-1]
    ind_silence3.insert(0 ,ind_silence[0])
    ind_silence = np.array(ind_silence)
    ind_silence2 = np.array(ind_silence2)
    ind_silence3 = np.array(ind_silence3)
    ind_silence.shape , ind_silence2.shape, ind_silence3.shape
    # if mod:
    #   ind_new =ind_silence[(abs(ind_silence - ind_silence2) > sr) | ( (ind_silence - ind_silence3 ) > sr)] 
    # else:
    #   ind_new =ind_silence

    ind_new =ind_silence[(abs(ind_silence - ind_silence2) > sr) | ( (ind_silence - ind_silence3 ) > sr)] 
    if not any(ind_new):
      ind_new =[ind_silence[np.argmax(abs(ind_silence - ind_silence2))],ind_silence2[np.argmax(abs(ind_silence - ind_silence2))]]
    # print('ind_new:',ind_new)
    sil_start = ind_new[::2];
    sil_stop = ind_new[1::2];
    l = min(len(sil_start) , len(sil_stop));
    return sil_start,sil_stop,l

def filter_bank_function(signal,sample_rate):
    pre_emphasis = 0.97
    # signal = signal_main2
    emphasized_signal = np.append(signal[0], signal[1:] - pre_emphasis * signal[:-1])
    frame_size = 0.025
    frame_stride = 0.01
    frame_length, frame_step = frame_size * sample_rate, frame_stride * sample_rate  # Convert from seconds to samples
    signal_length = len(emphasized_signal)
    frame_length = int(round(frame_length))
    frame_step = int(round(frame_step))
    num_frames = int(np.ceil(float(np.abs(signal_length - frame_length)) / frame_step))  # Make sure that we have at least 1 frame

    pad_signal_length = num_frames * frame_step + frame_length
    z = np.zeros((pad_signal_length - signal_length))
    pad_signal = np.append(emphasized_signal, z) # Pad Signal to make sure that all frames have equal number of samples without truncating any samples from the original signal

    indices = np.tile(np.arange(0, frame_length), (num_frames, 1)) + np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_length, 1)).T
    frames = pad_signal[indices.astype(np.int32, copy=False)]
    frames *= np.hamming(frame_length)
    NFFT = 512
    # frames *= 0.54 - 0.46 * np.cos((2 * np.pi * n) / (frame_length - 1))  # Explicit Implementation **
    mag_frames = np.absolute(np.fft.rfft(frames, NFFT))  # Magnitude of the FFT
    pow_frames = ((1.0 / NFFT) * ((mag_frames) ** 2))  # Power Spectrum
    nfilt = 26
    low_freq_mel = 0
    high_freq_mel = (2595 * np.log10(1 + (sample_rate / 2) / 700))  # Convert Hz to Mel
    mel_points = np.linspace(low_freq_mel, high_freq_mel, nfilt + 2)  # Equally spaced in Mel scale
    hz_points = (700 * (10**(mel_points / 2595) - 1))  # Convert Mel to Hz
    bin = np.floor((NFFT + 1) * hz_points / sample_rate)

    fbank = np.zeros((nfilt, int(np.floor(NFFT / 2 + 1))))
    for m in range(1, nfilt + 1):
        f_m_minus = int(bin[m - 1])   # left
        f_m = int(bin[m])             # center
        f_m_plus = int(bin[m + 1])    # right

        for k in range(f_m_minus, f_m):
            fbank[m - 1, k] = (k - bin[m - 1]) / (bin[m] - bin[m - 1])
        for k in range(f_m, f_m_plus):
            fbank[m - 1, k] = (bin[m + 1] - k) / (bin[m + 1] - bin[m])
    filter_banks = np.dot(pow_frames, fbank.T)
    filter_banks = np.where(filter_banks == 0, np.finfo(float).eps, filter_banks)  # Numerical Stability
    filter_banks = 20 * np.log10(filter_banks)  # dB
    filter_banks -= (np.mean(filter_banks, axis=0) + 1e-8)
    return filter_banks

def manipulate(data, speed_factor):
    return librosa.effects.time_stretch(data, speed_factor)


## Feature Extraction loop - Data Augmentation

In [ ]:
plt.close('all')
df_train = pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/train_csv.csv')
df_train.set_index('names',inplace=True)

folder_path = '/content/drive/My Drive/Speech project - Alzheimer/dataset/all_files/' #C:/Users/jalaledin/Desktop/Project Alzheimer/dataset/cnm/'
all_files = os.listdir(folder_path)
cnt = 1
failed_files=[]
for file0 in all_files[78:]:
  try:
    # file0 = 'adrso027.wav'
    # if file0 == 'adrso027.wav':
    # MFCCs = filter_banks = silence_percentage = sil_start = sil_stop = []
    signal_main, sample_rate = librosa.load(folder_path + file0)
    sr = sample_rate
    print(folder_path + file0)
    # print(df_train.loc[file0]['ad/cn'])
    consider_text_file = 0
    if consider_text_file:
      if df_train.loc[file0]['ad/cn'] == 1:
        class_path = 'ad'
      else:
        class_path = 'cn'

      txt_path = '/content/drive/My Drive/Speech project - Alzheimer/dataset/time/' + class_path + '/' + file0[:-3] + 'csv'
      try:
        txt_csv = pd.read_csv(txt_path)   
        txt_csv = txt_csv[txt_csv['speaker']=='PAR']
        begin_samp = list(txt_csv['begin'])
        end_samp = list(txt_csv['end'])
        signal_main_new =[]
        for k,x in enumerate(begin_samp):
            signal_main_new.append(signal_main[begin_samp[k]:end_samp[k]+1])
        signal_main = signal_main_new
        # print(txt_csv)
      except:
        print('**************************'+file0[:-3]+'********************************')

    FIG_SIZE = (10,4)
    # WAVEFORM
    # display waveform
    plt.figure(1,figsize=FIG_SIZE)
    librosa.display.waveplot(signal_main, sample_rate, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title("Waveform")

    signal_env = np.array(getEnvelope(signal_main))
    plt.figure(2 , figsize=FIG_SIZE)
    plt.plot( signal_env , 'b')

    guard_samp = np.int(np.floor(10*sr)) #10 sec gaurd from start and stop 
    if consider_text_file==1 or len(signal_env)<2*guard_samp+10*sr:
      guard_samp = 0

    signal_main2 = signal_main[guard_samp:-1-guard_samp]
    signal_env = signal_env[guard_samp:-1-guard_samp]
    signal_env = signal_env.T

    # plt.figure(1)
    # plt.plot(np.arange(guard_samp,len(signal0)-guard_samp-1)/(sr)  , signal , 'b');
    plt.figure(3 , figsize=FIG_SIZE)
    plt.plot( signal_env , 'b')

    # select section of data that is noise
    try :
      sil_start , sil_stop , l = silence_detection(signal_env,1)
      noisy_part = signal_main2[sil_start[0]:sil_stop[0]]
      for i in range(len(sil_start)-1):
        noisy_part = np.concatenate([noisy_part,signal_main2[sil_start[i+1]:sil_stop[i+1]]])
    except:
      sil_start , sil_stop , l = silence_detection(signal_env,0)
      noisy_part = signal_main2[sil_start[0]:sil_stop[0]]
      for i in range(len(sil_start)-2):
        noisy_part = np.concatenate([noisy_part,signal_main2[sil_start[i+1]:sil_stop[i+1]]])

    #=============================================================================================
    first_samp = [0, 19*sr, 39*sr, 59*sr, 79*sr, 99*sr, 119*sr, 139*sr, 159*sr, 179*sr, 199*sr, 219*sr, 239*sr, 259*sr, 279*sr, 299*sr]
    last_samp = [20*sr, 40*sr, 60*sr, 80*sr, 100*sr, 120*sr, 140*sr, 160*sr, 180*sr, 200*sr, 220*sr, 240*sr, 260*sr, 280*sr, 300*sr]
    segment_cnt=0
    for i,j in zip(first_samp,last_samp) :
      try:
        if len(signal_main2)<i:
          break
        if len(signal_main2)-j<10*sr or len(signal_main2)<j : 
          j = -1
        segment_cnt = segment_cnt+1

        signal_partial = signal_main2[i:j]
      
        # perform noise reduction
        reduced_noise = nr.reduce_noise(y=signal_partial, y_noise=noisy_part ,sr=sr )
        # reduced_noise = nr.reduce_noise(audio_clip=signal_partial, noise_clip=noisy_part, verbose=True)
        plt.figure(3 , figsize=(25,14))
        plt.plot(signal_partial,'b')
        plt.plot(reduced_noise,'r')
        plt.show()
        #=============================================================================================

        signal_env = np.array(getEnvelope(reduced_noise))
        signal_env = signal_env.T

        sil_start , sil_stop , l = silence_detection(signal_env,1)

        try:
          silence_length = sum(sil_stop - sil_start)/sr
        except:
          silence_length = (sil_stop[0] - sil_start[0])/sr
        sig_len = len(signal_env)/sr
        silence_percentage = silence_length/sig_len*100
        #sig_len , silence_length , silence_length/sig_len*100
        print('Silence length is {} seconds'.format(silence_length))
        print('Silence percentage is %{}'.format(silence_percentage))

        filter_banks = filter_bank_function(signal_partial , sample_rate)

        # STFT -> spectrogram
        hop_length = 512 # in num. of samples
        n_fft = 2048 # window in num. of samples

        # MFCCs
        MFCCs = librosa.feature.mfcc(signal_partial, sample_rate, n_fft=int(n_fft), hop_length=int(hop_length),n_mfcc=42)

        # display MFCCs
        plt.figure(figsize=FIG_SIZE)
        librosa.display.specshow(MFCCs, sr=sample_rate, hop_length=hop_length)
        plt.xlabel("Time")
        plt.ylabel("MFCC coefficients")
        plt.colorbar()
        plt.title("MFCCs")
        print(MFCCs.shape)

        # show plots
        plt.show()

        #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
        write('/content/drive/My Drive/Speech project - Alzheimer/dataset/test.wav', sample_rate, signal_partial)
        sound = Waveform(path='/content/drive/My Drive/Speech project - Alzheimer/dataset/test.wav' , sample_rate=sample_rate)

        shimmers = sound.shimmers()
        jitters = sound.jitters()
        HNR = sound.hnr()
        
        signal_main3 = copy.copy(signal_partial)
        while len(signal_main3)>1500000:
          signal_main3 = signal_main3[::2]
        
        # signal_main3 = signal_main3 / np.mean(sorted(signal_main3)[-500:]) #round(0.01*len(signal_main3))
        signal_main3 = signal_main3 - np.mean(signal_main3)
        # signal_main_data[file0] = [len(signal_main3) , max(signal_main3) , np.mean(signal_main3) , np.mean(sorted(signal_main3)[-500:]) , np.mean(sorted(signal_main3)[-round(0.01*len(signal_main3)):])]
        auto_cor = np.correlate(signal_main3, signal_main3, mode='full')
        auto_cor = auto_cor / max(auto_cor)
        print('***********',len(signal_main2))

        # df_features.at[file0,'jitter'] = jitters['localJitter']
        # df_features.at[file0,'shimmer'] = shimmers['localdbShimmer']
        # df_features.at[file0,'hnr'] = HNR
        # df_features.at[file0,'auto_corr'] = np.mean(auto_cor)
        #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

        data_vec = features(MFCCs , filter_banks , silence_percentage , sil_start , sil_stop , jitters['localJitter'] , shimmers['localdbShimmer'] , HNR , np.mean(auto_cor))
        if cnt==1:
          # data_vec.insert(0,file0+'_'+str(segment_cnt))
          # df_features = pd.DataFrame(np.array(data_vec).reshape(1,-1), columns = ['ID' , 'MFCC2' , 'kurt_MFCC30' , 'mean_MFCC30' , 'skew_MFCC2' , 'mean_MFCC16' , 'flt_bnk_eng22' , 'MFCC30' , 'kurt_MFCC16' , 'flt_bnk_eng2' , 'flt_bnk_eng24' , 'MFCC1' , 'flt_bnk_eng15' , 'kurt_MFCC2' , 'flt_bnk_eng20' , 'flt_bnk_eng13' , 'n_sil_segments' , 'frac_silence' , 'min_sil_len' , 'jitter' , 'shimmer' , 'HNR' , 'auto_correlation'])
          # cnt = 0

          df_features = pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented2.csv')
          # df_features.set_index('ID',inplace=True)
          # df_features = df_features.drop(columns=['Unnamed: 0'])
          cnt = 0
          new_entry = {'ID':[file0+'_'+str(segment_cnt)] , 'MFCC2':data_vec[0] , 'kurt_MFCC30':data_vec[1] , 'mean_MFCC30':data_vec[2] , 'skew_MFCC2':data_vec[3] , 'mean_MFCC16':data_vec[4] , 'flt_bnk_eng22':data_vec[5] , 'MFCC30':data_vec[6] , 'kurt_MFCC16':data_vec[7] , 'flt_bnk_eng2':data_vec[8] , 'flt_bnk_eng24':data_vec[9] , 'MFCC1':data_vec[10] , 'flt_bnk_eng15':data_vec[11] , 'kurt_MFCC2':data_vec[12] , 'flt_bnk_eng20':data_vec[13] , 'flt_bnk_eng13':data_vec[14] , 'n_sil_segments':data_vec[15] , 'frac_silence':data_vec[16] , 'min_sil_len':data_vec[17] , 'jitter':data_vec[18] , 'shimmer':data_vec[19] , 'HNR':data_vec[20] , 'auto_correlation':data_vec[21]}
          new_entry_df=pd.DataFrame.from_dict(new_entry)
          df_features = df_features.append(new_entry_df,ignore_index=True)
          # df_features.loc[file0] = np.array(data_vec).reshape(1,-1) #pd.DataFrame(data_vec)
          display(df_features)
        else:
          new_entry = {'ID':[file0+'_'+str(segment_cnt)] , 'MFCC2':data_vec[0] , 'kurt_MFCC30':data_vec[1] , 'mean_MFCC30':data_vec[2] , 'skew_MFCC2':data_vec[3] , 'mean_MFCC16':data_vec[4] , 'flt_bnk_eng22':data_vec[5] , 'MFCC30':data_vec[6] , 'kurt_MFCC16':data_vec[7] , 'flt_bnk_eng2':data_vec[8] , 'flt_bnk_eng24':data_vec[9] , 'MFCC1':data_vec[10] , 'flt_bnk_eng15':data_vec[11] , 'kurt_MFCC2':data_vec[12] , 'flt_bnk_eng20':data_vec[13] , 'flt_bnk_eng13':data_vec[14] , 'n_sil_segments':data_vec[15] , 'frac_silence':data_vec[16] , 'min_sil_len':data_vec[17] , 'jitter':data_vec[18] , 'shimmer':data_vec[19] , 'HNR':data_vec[20] , 'auto_correlation':data_vec[21]}
          new_entry_df=pd.DataFrame.from_dict(new_entry)
          df_features = df_features.append(new_entry_df,ignore_index=True)
          # df_features.loc[file0] = np.array(data_vec).reshape(1,-1) #pd.DataFrame(data_vec)
        display(df_features)
        df_features.to_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented3.csv')
      except:
        print('segment:',[file0+'_'+str(segment_cnt)] ,'failed.')
        failed_files.append([file0+'_'+str(segment_cnt)])
        with open("/content/drive/My Drive/Speech project - Alzheimer/file.txt", "w") as output:
          output.write(str(failed_files))
  except:
      print('file:' , file0 , 'failed.')
      failed_files.append([file0])
      with open("/content/drive/My Drive/Speech project - Alzheimer/file.txt", "w") as output:
        output.write(str(failed_files))
      

df_features.set_index('ID',inplace=True)
df_features.to_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented3.csv')
df_features

In [ ]:
plt.close('all')
df_train = pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/train_csv.csv')
df_train.set_index('names',inplace=True)

folder_path = '/content/drive/My Drive/Speech project - Alzheimer/dataset/all_files/' #C:/Users/jalaledin/Desktop/Project Alzheimer/dataset/cnm/'
all_files = os.listdir(folder_path)
cnt = 1
failed_files=[]
for file0 in all_files[53:]:
  try:
    # file0 = 'adrso027.wav'
    # if file0 == 'adrso027.wav':
    # MFCCs = filter_banks = silence_percentage = sil_start = sil_stop = []
    signal_main, sample_rate = librosa.load(folder_path + file0)
    sr = sample_rate
    print(folder_path + file0)
    # print(df_train.loc[file0]['ad/cn'])
    consider_text_file = 0
    if consider_text_file:
      if df_train.loc[file0]['ad/cn'] == 1:
        class_path = 'ad'
      else:
        class_path = 'cn'

      txt_path = '/content/drive/My Drive/Speech project - Alzheimer/dataset/time/' + class_path + '/' + file0[:-3] + 'csv'
      try:
        txt_csv = pd.read_csv(txt_path)   
        txt_csv = txt_csv[txt_csv['speaker']=='PAR']
        begin_samp = list(txt_csv['begin'])
        end_samp = list(txt_csv['end'])
        signal_main_new =[]
        for k,x in enumerate(begin_samp):
            signal_main_new.append(signal_main[begin_samp[k]:end_samp[k]+1])
        signal_main = signal_main_new
        # print(txt_csv)
      except:
        print('**************************'+file0[:-3]+'********************************')

    FIG_SIZE = (10,4)
    # WAVEFORM
    # display waveform
    plt.figure(1,figsize=FIG_SIZE)
    librosa.display.waveplot(signal_main, sample_rate, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title("Waveform")

    signal_env = np.array(getEnvelope(signal_main))
    plt.figure(2 , figsize=FIG_SIZE)
    plt.plot( signal_env , 'b')

    guard_samp = np.int(np.floor(10*sr)) #10 sec gaurd from start and stop 
    if consider_text_file==1 or len(signal_env)<2*guard_samp+10*sr:
      guard_samp = 0

    signal_main2 = signal_main[guard_samp:-1-guard_samp]
    signal_env = signal_env[guard_samp:-1-guard_samp]
    signal_env = signal_env.T

    # plt.figure(1)
    # plt.plot(np.arange(guard_samp,len(signal0)-guard_samp-1)/(sr)  , signal , 'b');
    plt.figure(3 , figsize=FIG_SIZE)
    plt.plot( signal_env , 'b')

    # select section of data that is noise
    try :
      sil_start , sil_stop , l = silence_detection(signal_env,1)
      noisy_part = signal_main2[sil_start[0]:sil_stop[0]]
      for i in range(len(sil_start)-1):
        noisy_part = np.concatenate([noisy_part,signal_main2[sil_start[i+1]:sil_stop[i+1]]])
    except:
      sil_start , sil_stop , l = silence_detection(signal_env,0)
      noisy_part = signal_main2[sil_start[0]:sil_stop[0]]
      for i in range(len(sil_start)-2):
        noisy_part = np.concatenate([noisy_part,signal_main2[sil_start[i+1]:sil_stop[i+1]]])

    #=============================================================================================
    a1 = 15
    first_samp = [0, (a1-1)*sr, (2*a1-1)*sr, (3*a1-1)*sr, (4*a1-1)*sr, (5*a1-1)*sr, (6*a1-1)*sr, (7*a1-1)*sr, (8*a1-1)*sr, (9*a1-1)*sr, (10*a1-1)*sr, (11*a1-1)*sr, (12*a1-1)*sr, (13*a1-1)*sr, (14*a1-1)*sr, (15*a1-1)*sr]
    last_samp = [a1*sr, 2*a1*sr, 3*a1*sr, 4*a1*sr, 5*a1*sr, 6*a1*sr, 7*a1*sr, 8*a1*sr, 9*a1*sr, 10*a1*sr, 11*a1*sr, 12*a1*sr, 13*a1*sr, 14*a1*sr, 15*a1*sr]
    segment_cnt=0
    for i,j in zip(first_samp,last_samp) :
      try:
        if len(signal_main2)<i:
          break
        if len(signal_main2)-j<10*sr or len(signal_main2)<j : 
          j = -1
        segment_cnt = segment_cnt+1

        signal_partial = signal_main2[i:j]
      
        # perform noise reduction
        reduced_noise = nr.reduce_noise(y=signal_partial, y_noise=noisy_part ,sr=sr )
        # reduced_noise = nr.reduce_noise(audio_clip=signal_partial, noise_clip=noisy_part, verbose=True)
        plt.figure(3 , figsize=(25,14))
        plt.plot(signal_partial,'b')
        plt.plot(reduced_noise,'r')
        plt.show()
        #=============================================================================================

        signal_env = np.array(getEnvelope(reduced_noise))
        signal_env = signal_env.T

        sil_start , sil_stop , l = silence_detection(signal_env,1)

        try:
          silence_length = sum(sil_stop - sil_start)/sr
        except:
          silence_length = (sil_stop[0] - sil_start[0])/sr
        sig_len = len(signal_env)/sr
        silence_percentage = silence_length/sig_len*100
        #sig_len , silence_length , silence_length/sig_len*100
        print('Silence length is {} seconds'.format(silence_length))
        print('Silence percentage is %{}'.format(silence_percentage))

        filter_banks = filter_bank_function(signal_partial , sample_rate)

        # STFT -> spectrogram
        hop_length = 512 # in num. of samples
        n_fft = 2048 # window in num. of samples

        # MFCCs
        MFCCs = librosa.feature.mfcc(signal_partial, sample_rate, n_fft=int(n_fft), hop_length=int(hop_length),n_mfcc=42)

        # display MFCCs
        plt.figure(figsize=FIG_SIZE)
        librosa.display.specshow(MFCCs, sr=sample_rate, hop_length=hop_length)
        plt.xlabel("Time")
        plt.ylabel("MFCC coefficients")
        plt.colorbar()
        plt.title("MFCCs")
        print(MFCCs.shape)

        # show plots
        plt.show()

        #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
        write('/content/drive/My Drive/Speech project - Alzheimer/dataset/test.wav', sample_rate, signal_partial)
        sound = Waveform(path='/content/drive/My Drive/Speech project - Alzheimer/dataset/test.wav' , sample_rate=sample_rate)

        shimmers = sound.shimmers()
        jitters = sound.jitters()
        HNR = sound.hnr()
        
        signal_main3 = copy.copy(signal_partial)
        while len(signal_main3)>1500000:
          signal_main3 = signal_main3[::2]
        
        # signal_main3 = signal_main3 / np.mean(sorted(signal_main3)[-500:]) #round(0.01*len(signal_main3))
        signal_main3 = signal_main3 - np.mean(signal_main3)
        # signal_main_data[file0] = [len(signal_main3) , max(signal_main3) , np.mean(signal_main3) , np.mean(sorted(signal_main3)[-500:]) , np.mean(sorted(signal_main3)[-round(0.01*len(signal_main3)):])]
        auto_cor = np.correlate(signal_main3, signal_main3, mode='full')
        auto_cor = auto_cor / max(auto_cor)
        print('***********',len(signal_main2))

        # df_features.at[file0,'jitter'] = jitters['localJitter']
        # df_features.at[file0,'shimmer'] = shimmers['localdbShimmer']
        # df_features.at[file0,'hnr'] = HNR
        # df_features.at[file0,'auto_corr'] = np.mean(auto_cor)
        #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

        data_vec = features(MFCCs , filter_banks , silence_percentage , sil_start , sil_stop , jitters['localJitter'] , shimmers['localdbShimmer'] , HNR , np.mean(auto_cor))
        if cnt==1:
          # data_vec.insert(0,file0+'_'+str(segment_cnt))
          # df_features = pd.DataFrame(np.array(data_vec).reshape(1,-1), columns = ['ID' , 'MFCC2' , 'kurt_MFCC30' , 'mean_MFCC30' , 'skew_MFCC2' , 'mean_MFCC16' , 'flt_bnk_eng22' , 'MFCC30' , 'kurt_MFCC16' , 'flt_bnk_eng2' , 'flt_bnk_eng24' , 'MFCC1' , 'flt_bnk_eng15' , 'kurt_MFCC2' , 'flt_bnk_eng20' , 'flt_bnk_eng13' , 'n_sil_segments' , 'frac_silence' , 'min_sil_len' , 'jitter' , 'shimmer' , 'HNR' , 'auto_correlation'])
          # cnt = 0

          df_features = pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented_15sec.csv')
          # df_features.set_index('ID',inplace=True)
          # df_features = df_features.drop(columns=['Unnamed: 0'])
          cnt = 0
          new_entry = {'ID':[file0+'_'+str(segment_cnt)] , 'MFCC2':data_vec[0] , 'kurt_MFCC30':data_vec[1] , 'mean_MFCC30':data_vec[2] , 'skew_MFCC2':data_vec[3] , 'mean_MFCC16':data_vec[4] , 'flt_bnk_eng22':data_vec[5] , 'MFCC30':data_vec[6] , 'kurt_MFCC16':data_vec[7] , 'flt_bnk_eng2':data_vec[8] , 'flt_bnk_eng24':data_vec[9] , 'MFCC1':data_vec[10] , 'flt_bnk_eng15':data_vec[11] , 'kurt_MFCC2':data_vec[12] , 'flt_bnk_eng20':data_vec[13] , 'flt_bnk_eng13':data_vec[14] , 'n_sil_segments':data_vec[15] , 'frac_silence':data_vec[16] , 'min_sil_len':data_vec[17] , 'jitter':data_vec[18] , 'shimmer':data_vec[19] , 'HNR':data_vec[20] , 'auto_correlation':data_vec[21]}
          new_entry_df=pd.DataFrame.from_dict(new_entry)
          df_features = df_features.append(new_entry_df,ignore_index=True)
          # df_features.loc[file0] = np.array(data_vec).reshape(1,-1) #pd.DataFrame(data_vec)
          display(df_features)
        else:
          new_entry = {'ID':[file0+'_'+str(segment_cnt)] , 'MFCC2':data_vec[0] , 'kurt_MFCC30':data_vec[1] , 'mean_MFCC30':data_vec[2] , 'skew_MFCC2':data_vec[3] , 'mean_MFCC16':data_vec[4] , 'flt_bnk_eng22':data_vec[5] , 'MFCC30':data_vec[6] , 'kurt_MFCC16':data_vec[7] , 'flt_bnk_eng2':data_vec[8] , 'flt_bnk_eng24':data_vec[9] , 'MFCC1':data_vec[10] , 'flt_bnk_eng15':data_vec[11] , 'kurt_MFCC2':data_vec[12] , 'flt_bnk_eng20':data_vec[13] , 'flt_bnk_eng13':data_vec[14] , 'n_sil_segments':data_vec[15] , 'frac_silence':data_vec[16] , 'min_sil_len':data_vec[17] , 'jitter':data_vec[18] , 'shimmer':data_vec[19] , 'HNR':data_vec[20] , 'auto_correlation':data_vec[21]}
          new_entry_df=pd.DataFrame.from_dict(new_entry)
          df_features = df_features.append(new_entry_df,ignore_index=True)
          # df_features.loc[file0] = np.array(data_vec).reshape(1,-1) #pd.DataFrame(data_vec)
        display(df_features)
        df_features.to_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented_15sec.csv')
      except:
        print('segment:',[file0+'_'+str(segment_cnt)] ,'failed.')
        failed_files.append([file0+'_'+str(segment_cnt)])
        with open("/content/drive/My Drive/Speech project - Alzheimer/file.txt", "w") as output:
          output.write(str(failed_files))
  except:
      print('file:' , file0 , 'failed.')
      failed_files.append([file0])
      with open("/content/drive/My Drive/Speech project - Alzheimer/file.txt", "w") as output:
        output.write(str(failed_files))
      

df_features.set_index('ID',inplace=True)
df_features.to_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented_15sec.csv')
df_features

## Features + Changing Speed for stages

In [ ]:
plt.close('all')
df_train = pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/train_csv.csv')
df_train.set_index('names',inplace=True)

df_me = pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/dataset_total333.csv')
df_me.set_index('ID' , inplace=True)

folder_path = '/content/drive/My Drive/Speech project - Alzheimer/dataset/all_files/' #C:/Users/jalaledin/Desktop/Project Alzheimer/dataset/cnm/'
all_files = os.listdir(folder_path)
cnt = 1
failed_files=[]
for file0 in all_files:
  if df_me.loc[file0]['Stage class']!=0:
    try:
      # file0 = 'adrso027.wav'
      # if file0 == 'adrso027.wav':
      # MFCCs = filter_banks = silence_percentage = sil_start = sil_stop = []
      signal_main, sample_rate = librosa.load(folder_path + file0)
      for speed_factor in [0.8 , 1.2]:
        signal_main = manipulate(signal_main, speed_factor)
        manipulate(signal_main, speed_factor)
        sr = sample_rate
        print(folder_path + file0)
        # print(df_train.loc[file0]['ad/cn'])
        consider_text_file = 0
        if consider_text_file:
          if df_train.loc[file0]['ad/cn'] == 1:
            class_path = 'ad'
          else:
            class_path = 'cn'

          txt_path = '/content/drive/My Drive/Speech project - Alzheimer/dataset/time/' + class_path + '/' + file0[:-3] + 'csv'
          try:
            txt_csv = pd.read_csv(txt_path)   
            txt_csv = txt_csv[txt_csv['speaker']=='PAR']
            begin_samp = list(txt_csv['begin'])
            end_samp = list(txt_csv['end'])
            signal_main_new =[]
            for k,x in enumerate(begin_samp):
                signal_main_new.append(signal_main[begin_samp[k]:end_samp[k]+1])
            signal_main = signal_main_new
            # print(txt_csv)
          except:
            print('**************************'+file0[:-3]+'********************************')

        FIG_SIZE = (10,4)
        # WAVEFORM
        # display waveform
        plt.figure(1,figsize=FIG_SIZE)
        librosa.display.waveplot(signal_main, sample_rate, alpha=0.4)
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude")
        plt.title("Waveform")

        signal_env = np.array(getEnvelope(signal_main))
        plt.figure(2 , figsize=FIG_SIZE)
        plt.plot( signal_env , 'b')

        guard_samp = np.int(np.floor(10*sr)) #10 sec gaurd from start and stop 
        if consider_text_file==1 or len(signal_env)<2*guard_samp+10*sr:
          guard_samp = 0

        signal_main2 = signal_main[guard_samp:-1-guard_samp]
        signal_env = signal_env[guard_samp:-1-guard_samp]
        signal_env = signal_env.T

        # plt.figure(1)
        # plt.plot(np.arange(guard_samp,len(signal0)-guard_samp-1)/(sr)  , signal , 'b');
        plt.figure(3 , figsize=FIG_SIZE)
        plt.plot( signal_env , 'b')

        # select section of data that is noise
        try :
          sil_start , sil_stop , l = silence_detection(signal_env,1)
          noisy_part = signal_main2[sil_start[0]:sil_stop[0]]
          for i in range(len(sil_start)-1):
            noisy_part = np.concatenate([noisy_part,signal_main2[sil_start[i+1]:sil_stop[i+1]]])
        except:
          sil_start , sil_stop , l = silence_detection(signal_env,0)
          noisy_part = signal_main2[sil_start[0]:sil_stop[0]]
          for i in range(len(sil_start)-2):
            noisy_part = np.concatenate([noisy_part,signal_main2[sil_start[i+1]:sil_stop[i+1]]])

        #=============================================================================================
        first_samp = [0, 19*sr, 39*sr, 59*sr, 79*sr, 99*sr, 119*sr, 139*sr, 159*sr, 179*sr, 199*sr, 219*sr, 239*sr, 259*sr, 279*sr, 299*sr]
        last_samp = [20*sr, 40*sr, 60*sr, 80*sr, 100*sr, 120*sr, 140*sr, 160*sr, 180*sr, 200*sr, 220*sr, 240*sr, 260*sr, 280*sr, 300*sr]
        segment_cnt=0
        for i,j in zip(first_samp,last_samp) :
          try:
            if len(signal_main2)<i:
              break
            if len(signal_main2)-j<10*sr or len(signal_main2)<j : 
              j = -1
            segment_cnt = segment_cnt+1

            signal_partial = signal_main2[i:j]
          
            # perform noise reduction
            reduced_noise = nr.reduce_noise(y=signal_partial, y_noise=noisy_part ,sr=sr )
            # reduced_noise = nr.reduce_noise(audio_clip=signal_partial, noise_clip=noisy_part, verbose=True)
            plt.figure(3 , figsize=(25,14))
            plt.plot(signal_partial,'b')
            plt.plot(reduced_noise,'r')
            plt.show()
            #=============================================================================================

            signal_env = np.array(getEnvelope(reduced_noise))
            signal_env = signal_env.T

            sil_start , sil_stop , l = silence_detection(signal_env,1)

            try:
              silence_length = sum(sil_stop - sil_start)/sr
            except:
              silence_length = (sil_stop[0] - sil_start[0])/sr
            sig_len = len(signal_env)/sr
            silence_percentage = silence_length/sig_len*100
            #sig_len , silence_length , silence_length/sig_len*100
            print('Silence length is {} seconds'.format(silence_length))
            print('Silence percentage is %{}'.format(silence_percentage))

            filter_banks = filter_bank_function(signal_partial , sample_rate)

            # STFT -> spectrogram
            hop_length = 512 # in num. of samples
            n_fft = 2048 # window in num. of samples

            # MFCCs
            MFCCs = librosa.feature.mfcc(signal_partial, sample_rate, n_fft=int(n_fft), hop_length=int(hop_length),n_mfcc=42)

            # display MFCCs
            plt.figure(figsize=FIG_SIZE)
            librosa.display.specshow(MFCCs, sr=sample_rate, hop_length=hop_length)
            plt.xlabel("Time")
            plt.ylabel("MFCC coefficients")
            plt.colorbar()
            plt.title("MFCCs")
            print(MFCCs.shape)

            # show plots
            plt.show()

            #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
            write('/content/drive/My Drive/Speech project - Alzheimer/dataset/test.wav', sample_rate, signal_partial)
            sound = Waveform(path='/content/drive/My Drive/Speech project - Alzheimer/dataset/test.wav' , sample_rate=sample_rate)

            shimmers = sound.shimmers()
            jitters = sound.jitters()
            HNR = sound.hnr()
            
            signal_main3 = copy.copy(signal_partial)
            while len(signal_main3)>1500000:
              signal_main3 = signal_main3[::2]
            
            # signal_main3 = signal_main3 / np.mean(sorted(signal_main3)[-500:]) #round(0.01*len(signal_main3))
            signal_main3 = signal_main3 - np.mean(signal_main3)
            # signal_main_data[file0] = [len(signal_main3) , max(signal_main3) , np.mean(signal_main3) , np.mean(sorted(signal_main3)[-500:]) , np.mean(sorted(signal_main3)[-round(0.01*len(signal_main3)):])]
            auto_cor = np.correlate(signal_main3, signal_main3, mode='full')
            auto_cor = auto_cor / max(auto_cor)
            print('***********',len(signal_main2))

            # df_features.at[file0,'jitter'] = jitters['localJitter']
            # df_features.at[file0,'shimmer'] = shimmers['localdbShimmer']
            # df_features.at[file0,'hnr'] = HNR
            # df_features.at[file0,'auto_corr'] = np.mean(auto_cor)
            #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

            data_vec = features(MFCCs , filter_banks , silence_percentage , sil_start , sil_stop , jitters['localJitter'] , shimmers['localdbShimmer'] , HNR , np.mean(auto_cor))
            if cnt==1:
              # data_vec.insert(0,file0+'_'+str(segment_cnt))
              # df_features = pd.DataFrame(np.array(data_vec).reshape(1,-1), columns = ['ID' , 'MFCC2' , 'kurt_MFCC30' , 'mean_MFCC30' , 'skew_MFCC2' , 'mean_MFCC16' , 'flt_bnk_eng22' , 'MFCC30' , 'kurt_MFCC16' , 'flt_bnk_eng2' , 'flt_bnk_eng24' , 'MFCC1' , 'flt_bnk_eng15' , 'kurt_MFCC2' , 'flt_bnk_eng20' , 'flt_bnk_eng13' , 'n_sil_segments' , 'frac_silence' , 'min_sil_len' , 'jitter' , 'shimmer' , 'HNR' , 'auto_correlation'])
              # cnt = 0

              df_features = pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented3.csv')
              # df_features.set_index('ID',inplace=True)
              # df_features = df_features.drop(columns=['Unnamed: 0'])
              cnt = 0
              new_entry = {'ID':[file0+'+'+str(speed_factor)+'+'+str(segment_cnt)] , 'MFCC2':data_vec[0] , 'kurt_MFCC30':data_vec[1] , 'mean_MFCC30':data_vec[2] , 'skew_MFCC2':data_vec[3] , 'mean_MFCC16':data_vec[4] , 'flt_bnk_eng22':data_vec[5] , 'MFCC30':data_vec[6] , 'kurt_MFCC16':data_vec[7] , 'flt_bnk_eng2':data_vec[8] , 'flt_bnk_eng24':data_vec[9] , 'MFCC1':data_vec[10] , 'flt_bnk_eng15':data_vec[11] , 'kurt_MFCC2':data_vec[12] , 'flt_bnk_eng20':data_vec[13] , 'flt_bnk_eng13':data_vec[14] , 'n_sil_segments':data_vec[15] , 'frac_silence':data_vec[16] , 'min_sil_len':data_vec[17] , 'jitter':data_vec[18] , 'shimmer':data_vec[19] , 'HNR':data_vec[20] , 'auto_correlation':data_vec[21]}
              new_entry_df=pd.DataFrame.from_dict(new_entry)
              df_features = df_features.append(new_entry_df,ignore_index=True)
              # df_features.loc[file0] = np.array(data_vec).reshape(1,-1) #pd.DataFrame(data_vec)
              display(df_features)
            else:
              new_entry = {'ID':[file0+'+'+str(speed_factor)+'+'+str(segment_cnt)] , 'MFCC2':data_vec[0] , 'kurt_MFCC30':data_vec[1] , 'mean_MFCC30':data_vec[2] , 'skew_MFCC2':data_vec[3] , 'mean_MFCC16':data_vec[4] , 'flt_bnk_eng22':data_vec[5] , 'MFCC30':data_vec[6] , 'kurt_MFCC16':data_vec[7] , 'flt_bnk_eng2':data_vec[8] , 'flt_bnk_eng24':data_vec[9] , 'MFCC1':data_vec[10] , 'flt_bnk_eng15':data_vec[11] , 'kurt_MFCC2':data_vec[12] , 'flt_bnk_eng20':data_vec[13] , 'flt_bnk_eng13':data_vec[14] , 'n_sil_segments':data_vec[15] , 'frac_silence':data_vec[16] , 'min_sil_len':data_vec[17] , 'jitter':data_vec[18] , 'shimmer':data_vec[19] , 'HNR':data_vec[20] , 'auto_correlation':data_vec[21]}
              new_entry_df=pd.DataFrame.from_dict(new_entry)
              df_features = df_features.append(new_entry_df,ignore_index=True)
              # df_features.loc[file0] = np.array(data_vec).reshape(1,-1) #pd.DataFrame(data_vec)
            display(df_features)
            df_features.to_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented4.csv')
          except:
            print('segment:',[file0+'+'+str(speed_factor)+'+'+str(segment_cnt)] ,'failed.')
            failed_files.append([file0+'+'+str(speed_factor)+'+'+str(segment_cnt)])
            with open("/content/drive/My Drive/Speech project - Alzheimer/file.txt", "w") as output:
              output.write(str(failed_files))
    except:
        print('file:' , file0 , 'failed.')
        failed_files.append([file0])
        with open("/content/drive/My Drive/Speech project - Alzheimer/file.txt", "w") as output:
          output.write(str(failed_files))
        

df_features.set_index('ID',inplace=True)
df_features.to_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented4.csv')
df_features

## 0.8 , 1.2 , 1.3 , 0.7 Speed+Augmentation

In [ ]:
plt.close('all')
df_train = pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/train_csv.csv')
df_train.set_index('names',inplace=True)

df_me = pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/dataset_total333.csv')
df_me.set_index('ID' , inplace=True)

folder_path = '/content/drive/My Drive/Speech project - Alzheimer/dataset/all_files/' #C:/Users/jalaledin/Desktop/Project Alzheimer/dataset/cnm/'
all_files = os.listdir(folder_path)
cnt = 1
failed_files=[]
for file0 in all_files[137:]:
  if df_me.loc[file0]['Stage class']==0:
    ax = [0.7 , 0.8 , 1.2 , 1.3]
  else:
    ax = [0.7 , 1.3]
  try:
    # file0 = 'adrso027.wav'
    # if file0 == 'adrso027.wav':
    # MFCCs = filter_banks = silence_percentage = sil_start = sil_stop = []
    signal_main, sample_rate = librosa.load(folder_path + file0)
    for speed_factor in ax:
      signal_main = manipulate(signal_main, speed_factor)
      manipulate(signal_main, speed_factor)
      sr = sample_rate
      print(folder_path + file0)
      # print(df_train.loc[file0]['ad/cn'])
      consider_text_file = 0
      if consider_text_file:
        if df_train.loc[file0]['ad/cn'] == 1:
          class_path = 'ad'
        else:
          class_path = 'cn'

        txt_path = '/content/drive/My Drive/Speech project - Alzheimer/dataset/time/' + class_path + '/' + file0[:-3] + 'csv'
        try:
          txt_csv = pd.read_csv(txt_path)   
          txt_csv = txt_csv[txt_csv['speaker']=='PAR']
          begin_samp = list(txt_csv['begin'])
          end_samp = list(txt_csv['end'])
          signal_main_new =[]
          for k,x in enumerate(begin_samp):
              signal_main_new.append(signal_main[begin_samp[k]:end_samp[k]+1])
          signal_main = signal_main_new
          # print(txt_csv)
        except:
          print('**************************'+file0[:-3]+'********************************')

      FIG_SIZE = (10,4)
      # WAVEFORM
      # display waveform
      plt.figure(1,figsize=FIG_SIZE)
      librosa.display.waveplot(signal_main, sample_rate, alpha=0.4)
      plt.xlabel("Time (s)")
      plt.ylabel("Amplitude")
      plt.title("Waveform")

      signal_env = np.array(getEnvelope(signal_main))
      plt.figure(2 , figsize=FIG_SIZE)
      plt.plot( signal_env , 'b')

      guard_samp = np.int(np.floor(10*sr)) #10 sec gaurd from start and stop 
      if consider_text_file==1 or len(signal_env)<2*guard_samp+10*sr:
        guard_samp = 0

      signal_main2 = signal_main[guard_samp:-1-guard_samp]
      signal_env = signal_env[guard_samp:-1-guard_samp]
      signal_env = signal_env.T

      # plt.figure(1)
      # plt.plot(np.arange(guard_samp,len(signal0)-guard_samp-1)/(sr)  , signal , 'b');
      plt.figure(3 , figsize=FIG_SIZE)
      plt.plot( signal_env , 'b')

      # select section of data that is noise
      try :
        sil_start , sil_stop , l = silence_detection(signal_env,1)
        noisy_part = signal_main2[sil_start[0]:sil_stop[0]]
        for i in range(len(sil_start)-1):
          noisy_part = np.concatenate([noisy_part,signal_main2[sil_start[i+1]:sil_stop[i+1]]])
      except:
        sil_start , sil_stop , l = silence_detection(signal_env,0)
        noisy_part = signal_main2[sil_start[0]:sil_stop[0]]
        for i in range(len(sil_start)-2):
          noisy_part = np.concatenate([noisy_part,signal_main2[sil_start[i+1]:sil_stop[i+1]]])

      #=============================================================================================
      first_samp = [0, 19*sr, 39*sr, 59*sr, 79*sr, 99*sr, 119*sr, 139*sr, 159*sr, 179*sr, 199*sr, 219*sr, 239*sr, 259*sr, 279*sr, 299*sr]
      last_samp = [20*sr, 40*sr, 60*sr, 80*sr, 100*sr, 120*sr, 140*sr, 160*sr, 180*sr, 200*sr, 220*sr, 240*sr, 260*sr, 280*sr, 300*sr]
      segment_cnt=0
      for i,j in zip(first_samp,last_samp) :
        try:
          if len(signal_main2)<i:
            break
          if len(signal_main2)-j<10*sr or len(signal_main2)<j : 
            j = -1
          segment_cnt = segment_cnt+1

          signal_partial = signal_main2[i:j]
        
          # perform noise reduction
          reduced_noise = nr.reduce_noise(y=signal_partial, y_noise=noisy_part ,sr=sr )
          # reduced_noise = nr.reduce_noise(audio_clip=signal_partial, noise_clip=noisy_part, verbose=True)
          plt.figure(3 , figsize=(25,14))
          plt.plot(signal_partial,'b')
          plt.plot(reduced_noise,'r')
          plt.show()
          #=============================================================================================

          signal_env = np.array(getEnvelope(reduced_noise))
          signal_env = signal_env.T

          sil_start , sil_stop , l = silence_detection(signal_env,1)

          try:
            silence_length = sum(sil_stop - sil_start)/sr
          except:
            silence_length = (sil_stop[0] - sil_start[0])/sr
          sig_len = len(signal_env)/sr
          silence_percentage = silence_length/sig_len*100
          #sig_len , silence_length , silence_length/sig_len*100
          print('Silence length is {} seconds'.format(silence_length))
          print('Silence percentage is %{}'.format(silence_percentage))

          filter_banks = filter_bank_function(signal_partial , sample_rate)

          # STFT -> spectrogram
          hop_length = 512 # in num. of samples
          n_fft = 2048 # window in num. of samples

          # MFCCs
          MFCCs = librosa.feature.mfcc(signal_partial, sample_rate, n_fft=int(n_fft), hop_length=int(hop_length),n_mfcc=42)

          # display MFCCs
          plt.figure(figsize=FIG_SIZE)
          librosa.display.specshow(MFCCs, sr=sample_rate, hop_length=hop_length)
          plt.xlabel("Time")
          plt.ylabel("MFCC coefficients")
          plt.colorbar()
          plt.title("MFCCs")
          print(MFCCs.shape)

          # show plots
          plt.show()

          #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
          write('/content/drive/My Drive/Speech project - Alzheimer/dataset/test.wav', sample_rate, signal_partial)
          sound = Waveform(path='/content/drive/My Drive/Speech project - Alzheimer/dataset/test.wav' , sample_rate=sample_rate)

          shimmers = sound.shimmers()
          jitters = sound.jitters()
          HNR = sound.hnr()
          
          signal_main3 = copy.copy(signal_partial)
          while len(signal_main3)>1500000:
            signal_main3 = signal_main3[::2]
          
          # signal_main3 = signal_main3 / np.mean(sorted(signal_main3)[-500:]) #round(0.01*len(signal_main3))
          signal_main3 = signal_main3 - np.mean(signal_main3)
          # signal_main_data[file0] = [len(signal_main3) , max(signal_main3) , np.mean(signal_main3) , np.mean(sorted(signal_main3)[-500:]) , np.mean(sorted(signal_main3)[-round(0.01*len(signal_main3)):])]
          auto_cor = np.correlate(signal_main3, signal_main3, mode='full')
          auto_cor = auto_cor / max(auto_cor)
          print('***********',len(signal_main2))

          # df_features.at[file0,'jitter'] = jitters['localJitter']
          # df_features.at[file0,'shimmer'] = shimmers['localdbShimmer']
          # df_features.at[file0,'hnr'] = HNR
          # df_features.at[file0,'auto_corr'] = np.mean(auto_cor)
          #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

          data_vec = features(MFCCs , filter_banks , silence_percentage , sil_start , sil_stop , jitters['localJitter'] , shimmers['localdbShimmer'] , HNR , np.mean(auto_cor))
          if cnt==1:
            # data_vec.insert(0,file0+'_'+str(segment_cnt))
            # df_features = pd.DataFrame(np.array(data_vec).reshape(1,-1), columns = ['ID' , 'MFCC2' , 'kurt_MFCC30' , 'mean_MFCC30' , 'skew_MFCC2' , 'mean_MFCC16' , 'flt_bnk_eng22' , 'MFCC30' , 'kurt_MFCC16' , 'flt_bnk_eng2' , 'flt_bnk_eng24' , 'MFCC1' , 'flt_bnk_eng15' , 'kurt_MFCC2' , 'flt_bnk_eng20' , 'flt_bnk_eng13' , 'n_sil_segments' , 'frac_silence' , 'min_sil_len' , 'jitter' , 'shimmer' , 'HNR' , 'auto_correlation'])
            # cnt = 0

            df_features = pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented55.csv')
            # df_features.set_index('ID',inplace=True)
            # df_features = df_features.drop(columns=['Unnamed: 0'])
            cnt = 0
            new_entry = {'ID':[file0+'+'+str(speed_factor)+'+'+str(segment_cnt)] , 'MFCC2':data_vec[0] , 'kurt_MFCC30':data_vec[1] , 'mean_MFCC30':data_vec[2] , 'skew_MFCC2':data_vec[3] , 'mean_MFCC16':data_vec[4] , 'flt_bnk_eng22':data_vec[5] , 'MFCC30':data_vec[6] , 'kurt_MFCC16':data_vec[7] , 'flt_bnk_eng2':data_vec[8] , 'flt_bnk_eng24':data_vec[9] , 'MFCC1':data_vec[10] , 'flt_bnk_eng15':data_vec[11] , 'kurt_MFCC2':data_vec[12] , 'flt_bnk_eng20':data_vec[13] , 'flt_bnk_eng13':data_vec[14] , 'n_sil_segments':data_vec[15] , 'frac_silence':data_vec[16] , 'min_sil_len':data_vec[17] , 'jitter':data_vec[18] , 'shimmer':data_vec[19] , 'HNR':data_vec[20] , 'auto_correlation':data_vec[21]}
            new_entry_df=pd.DataFrame.from_dict(new_entry)
            df_features = df_features.append(new_entry_df,ignore_index=True)
            # df_features.loc[file0] = np.array(data_vec).reshape(1,-1) #pd.DataFrame(data_vec)
            display(df_features)
          else:
            new_entry = {'ID':[file0+'+'+str(speed_factor)+'+'+str(segment_cnt)] , 'MFCC2':data_vec[0] , 'kurt_MFCC30':data_vec[1] , 'mean_MFCC30':data_vec[2] , 'skew_MFCC2':data_vec[3] , 'mean_MFCC16':data_vec[4] , 'flt_bnk_eng22':data_vec[5] , 'MFCC30':data_vec[6] , 'kurt_MFCC16':data_vec[7] , 'flt_bnk_eng2':data_vec[8] , 'flt_bnk_eng24':data_vec[9] , 'MFCC1':data_vec[10] , 'flt_bnk_eng15':data_vec[11] , 'kurt_MFCC2':data_vec[12] , 'flt_bnk_eng20':data_vec[13] , 'flt_bnk_eng13':data_vec[14] , 'n_sil_segments':data_vec[15] , 'frac_silence':data_vec[16] , 'min_sil_len':data_vec[17] , 'jitter':data_vec[18] , 'shimmer':data_vec[19] , 'HNR':data_vec[20] , 'auto_correlation':data_vec[21]}
            new_entry_df=pd.DataFrame.from_dict(new_entry)
            df_features = df_features.append(new_entry_df,ignore_index=True)
            # df_features.loc[file0] = np.array(data_vec).reshape(1,-1) #pd.DataFrame(data_vec)
          display(df_features)
          df_features.to_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented55.csv')
        except:
          print('segment:',[file0+'+'+str(speed_factor)+'+'+str(segment_cnt)] ,'failed.')
          failed_files.append([file0+'+'+str(speed_factor)+'+'+str(segment_cnt)])
          with open("/content/drive/My Drive/Speech project - Alzheimer/file.txt", "w") as output:
            output.write(str(failed_files))
  except:
      print('file:' , file0 , 'failed.')
      failed_files.append([file0])
      with open("/content/drive/My Drive/Speech project - Alzheimer/file.txt", "w") as output:
        output.write(str(failed_files))
        

df_features.set_index('ID',inplace=True)
df_features.to_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented55.csv')
df_features

In [ ]:
df2 = pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented3.csv')
df2 = df2.drop(columns=['Unnamed: 0','Unnamed: 0.1'])
df2.set_index('ID',inplace=True)
df2.to_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented4.csv')
df2

In [ ]:
DS_tot = pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/dataset_total.csv')
DS_tot
ds_new = DS_tot[['ID','ad/cn_x','Stage class']]
ds_new
a = list(df2.index)
b = [x.split('_')[0] for x in a]
b
ds_new = ds_new.set_index('ID')
ad_cn = [ds_new.loc[x]['ad/cn_x'] for x in b]
stgs = [ds_new.loc[x]['Stage class'] for x in b]
stgs
df2['ad/cn'] = ad_cn
df2['Stage class'] = stgs
df2.to_csv('/content/drive/My Drive/Speech project - Alzheimer/features_segmented4.csv')
df2

In [ ]:
import pandas as pd
dataset =  pd.read_csv('/content/drive/My Drive/Speech project - Alzheimer/dataset_total.csv');
dataset.set_index('ID',inplace=True)
display(dataset)

## **Extracting New features: jitter, shimmer, hnr & autocorrelation**

# For 1 signal (testing)




In [ ]:
from surfboard.sound import Waveform
from surfboard.feature_extraction import extract_features
import librosa
import numpy as np
from scipy.io.wavfile import write

folder_path = '/content/drive/My Drive/Speech project - Alzheimer/dataset/all_files/' #C:/Users/jalaledin/Desktop/Project Alzheimer/dataset/cnm/'
file0 = 'adrso027.wav'
signal_main, sample_rate = librosa.load(folder_path + file0)
# sound = Waveform(path=folder_path + file0, sample_rate=sample_rate)

# plt.figure(1,figsize=FIG_SIZE)
# librosa.display.waveplot(signal_main, sample_rate, alpha=0.4)
# plt.xlabel("Time (s)")
# plt.ylabel("Amplitude")
# plt.title("Waveform")

# signal_env/ = np.array(getEnvelope(signal_main))
# plt.figure(2 , figsize=FIG_SIZE)
# plt.plot( signal_env , 'b')

guard_samp = np.int(np.floor(10*sample_rate)) #10 sec gaurd from start and stop 
# if consider_text_file==1 or len(signal_env)<2*guard_samp+10*sr:
#   guard_samp = 0

signal_main2 = signal_main[guard_samp:-1-guard_samp]

write('/content/drive/My Drive/Speech project - Alzheimer/dataset/test.wav', sample_rate, signal_main2)
sound = Waveform(path='/content/drive/My Drive/Speech project - Alzheimer/dataset/test.wav' , sample_rate=sample_rate)

shimmers = sound.shimmers()
jitters = sound.jitters()
HNR = sound.hnr()

# # type(signal_main)
# result = np.correlate(signal_main, signal_main, mode='full')

# shimmers['localdbShimmer'], jitters['localJitter'], HNR , np.mean(result)